In [1]:
import pandas as pd
df = pd.read_csv("data-mlt/MLT.csv")
dupes_mask = df.duplicated(subset=["SMILES", "cell_id"], keep=False)
assert not dupes_mask.any(), "Duplicate (SMILES, cell_id) pairs found!"

In [2]:
import logging
from pathlib import Path
import numpy as np
import pandas as pd
from rdkit import rdBase  
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from scipy.stats import pearsonr

from utils import (
    append_morgan_fingerprints_to_df,
    process_invalid_rows,
)


DATA_PATH = Path("data-mlt") / "MLT.csv"
RANDOM_STATE = 42
TEST_SIZE = 0.20    
N_BITS = 2048


USE_RIDGE = True                    
RIDGE_ALPHAS = np.logspace(-4, 3, 16)  
REFIT_ON_TRAINVAL = True           


def make_ohe():

    try:
        return OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    except TypeError:
        return OneHotEncoder(sparse=False, handle_unknown="ignore")


def print_split_info(n_total, idx_train, idx_val, idx_test):
    print(
        f"all: {n_total} | train: {len(idx_train)} | val: {len(idx_val)} | test: {len(idx_test)}"
    )


def _safe_pearson(x: np.ndarray, y: np.ndarray) -> float:
    x = np.ravel(x); y = np.ravel(y)
    if x.size == 0 or y.size == 0 or x.size != y.size:
        return np.nan
    if np.allclose(x, x[0]) or np.allclose(y, y[0]):  # 常量向量
        return np.nan
    try:
        return pearsonr(x, y)[0]
    except Exception:
        return np.nan


def _safe_r2_sample(y_true_row: np.ndarray, y_pred_row: np.ndarray, y_mean: np.ndarray) -> float:
    yt = np.ravel(y_true_row); yp = np.ravel(y_pred_row); ym = np.ravel(y_mean)
    if yt.size == 0 or yp.size == 0 or yt.size != yp.size or ym.size != yt.size:
        return np.nan
    sse = np.sum((yt - yp) ** 2)
    tss = np.sum((yt - ym) ** 2)
    if tss <= 0:
        return np.nan
    return 1.0 - sse / tss

def eval_and_print_samplewise(y_true: np.ndarray, y_pred: np.ndarray, name: str):

    if y_true.ndim == 1:
        y_true = y_true.reshape(-1, 1)
    if y_pred.ndim == 1:
        y_pred = y_pred.reshape(-1, 1)
    y_mean = np.mean(y_true, axis=0)
    per_sample_pearson = []
    per_sample_r2 = []
    for i in range(y_true.shape[0]):
        per_sample_pearson.append(_safe_pearson(y_true[i, :], y_pred[i, :]))
        per_sample_r2.append(_safe_r2_sample(y_true[i, :], y_pred[i, :], y_mean))
    per_sample_pearson = np.array(per_sample_pearson, dtype=float)
    per_sample_r2 = np.array(per_sample_r2, dtype=float)
    p_mean = np.nanmean(per_sample_pearson); p_med = np.nanmedian(per_sample_pearson); p_std = np.nanstd(per_sample_pearson)
    r_mean = np.nanmean(per_sample_r2);       r_med = np.nanmedian(per_sample_r2);       r_std = np.nanstd(per_sample_r2)

    print(
        f"[{name}] Pearson(sample mean/median/std) = {p_mean:.4f} / {p_med:.4f} / {p_std:.4f} | "
        f"R²(sample mean/median/std) = {r_mean:.4f} / {r_med:.4f} / {r_std:.4f}"
    )


def _samplewise_r2_mean(y_true: np.ndarray, y_pred: np.ndarray) -> float:

    if y_true.ndim == 1: y_true = y_true.reshape(-1, 1)
    if y_pred.ndim == 1: y_pred = y_pred.reshape(-1, 1)
    ym = np.mean(y_true, axis=0)
    vals = []
    for i in range(y_true.shape[0]):
        yt = y_true[i, :]; yp = y_pred[i, :]
        sse = np.sum((yt - yp) ** 2)
        tss = np.sum((yt - ym) ** 2)
        vals.append(np.nan if tss <= 0 else 1.0 - sse / tss)
    vals = np.array(vals, dtype=float)
    return float(np.nanmean(vals))


def tune_ridge_by_val(X_train, y_train, X_val, y_val, alphas):

    best_alpha, best_score, best_model = None, -np.inf, None
    for a in alphas:
        model = Ridge(alpha=a, random_state=RANDOM_STATE)
        model.fit(X_train, y_train)
        y_val_pred = model.predict(X_val)
        score = _samplewise_r2_mean(y_val, y_val_pred)
        if score > best_score:
            best_alpha, best_score, best_model = a, score, model
    logging.info(f"[Ridge Tuning] best alpha={best_alpha} | val samplewise R² mean={best_score:.4f}")
    return best_model, best_alpha, best_score


def main():
    np.random.seed(RANDOM_STATE)
    logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
    logger = logging.getLogger(__name__)

    # 1) 读入与指纹/清洗
    logger.info("Loading data...")
    df = pd.read_csv(DATA_PATH)

    df, _ = append_morgan_fingerprints_to_df(df)  
    df, _ = process_invalid_rows(df, True)         


    X = df.iloc[:, -N_BITS:].to_numpy()
    y = df.iloc[:, 2:-N_BITS].to_numpy()
    cell_id = df["cell_id"].astype(str)

    n_samples = len(df)
    idx_all = np.arange(n_samples)


    logger.info("Splitting data into tmp/test (80/20)...")
    (
        X_tmp, X_test,
        y_tmp, y_test,
        cell_id_tmp, cell_id_test,
        idx_tmp, idx_test,
    ) = train_test_split(
        X, y, cell_id, idx_all,
        test_size=TEST_SIZE,                 # 0.20
        random_state=RANDOM_STATE,
        shuffle=True,
    )
    
    logger.info("Splitting tmp into train/val (80/20)...")
    (
        X_train, X_val,
        y_train, y_val,
        cell_id_train, cell_id_val,
        idx_train, idx_val,
    ) = train_test_split(
        X_tmp, y_tmp, cell_id_tmp, idx_tmp,
        test_size=0.20,
        random_state=RANDOM_STATE,
        shuffle=True,
    )

    print_split_info(n_samples, idx_train, idx_val, idx_test)


    logger.info("Fitting OneHotEncoder on train cell_id...")
    ohe = make_ohe()
    cell_id_train_encoded = ohe.fit_transform(cell_id_train.values.reshape(-1, 1))
    cell_id_val_encoded   = ohe.transform(cell_id_val.values.reshape(-1, 1))
    cell_id_test_encoded  = ohe.transform(cell_id_test.values.reshape(-1, 1))
    ohe_dim = sum(len(cats) for cats in getattr(ohe, "categories_", []))
    logger.info(f"OHE categories: {ohe_dim} one-hot columns")


    X_train_combined = np.hstack((X_train, cell_id_train_encoded))
    X_val_combined   = np.hstack((X_val,   cell_id_val_encoded))
    X_test_combined  = np.hstack((X_test,  cell_id_test_encoded))


    logger.info("Computing 2nd/98th percentiles on train features/labels...")
    lower_bound_X = np.percentile(X_train_combined, 2, axis=0)
    upper_bound_X = np.percentile(X_train_combined, 98, axis=0)
    X_train_capped = np.clip(X_train_combined, lower_bound_X, upper_bound_X)
    X_val_capped   = np.clip(X_val_combined,   lower_bound_X, upper_bound_X)
    X_test_capped  = np.clip(X_test_combined,  lower_bound_X, upper_bound_X)

    lower_bound_y = np.percentile(y_train, 2, axis=0)
    upper_bound_y = np.percentile(y_train, 98, axis=0)
    y_train_capped = np.clip(y_train, lower_bound_y, upper_bound_y)
    y_val_capped   = np.clip(y_val,   lower_bound_y, upper_bound_y)
    y_test_capped  = np.clip(y_test,  lower_bound_y, upper_bound_y)


    if USE_RIDGE:
        logger.info("Tuning Ridge(alpha) on validation set...")

        tuned_model, best_alpha, best_val_score = tune_ridge_by_val(
            X_train_capped, y_train_capped, X_val_capped, y_val_capped, RIDGE_ALPHAS
        )

  
        if REFIT_ON_TRAINVAL:
            logger.info(f"Refitting final Ridge on Train+Val with alpha={best_alpha} ...")
            X_trval = np.vstack([X_train_capped, X_val_capped])
            y_trval = np.vstack([y_train_capped, y_val_capped])
            final_model = Ridge(alpha=best_alpha, random_state=RANDOM_STATE)
            final_model.fit(X_trval, y_trval)
        else:
            final_model = tuned_model

        y_val_pred  = tuned_model.predict(X_val_capped)    
        y_test_pred = final_model.predict(X_test_capped)   
    else:
        logger.info("Training Ordinary Least Squares (no hyperparameters)...")
        final_model = LinearRegression()
        final_model.fit(X_train_capped, y_train_capped)
        y_val_pred  = final_model.predict(X_val_capped)
        y_test_pred = final_model.predict(X_test_capped)


    eval_and_print_samplewise(y_val_capped,  y_val_pred,  name="Val")
    eval_and_print_samplewise(y_test_capped, y_test_pred, name="Test")


if __name__ == "__main__":
    main()


INFO: Loading data...
INFO: Splitting data into tmp/test (80/20)...
INFO: Splitting tmp into train/val (80/20)...
INFO: Fitting OneHotEncoder on train cell_id...
INFO: OHE categories: 4 one-hot columns
INFO: Computing 2nd/98th percentiles on train features/labels...
INFO: Tuning Ridge(alpha) on validation set...


all: 2438 | train: 1560 | val: 390 | test: 488


INFO: [Ridge Tuning] best alpha=39.81071705534969 | val samplewise R² mean=0.2903
INFO: Refitting final Ridge on Train+Val with alpha=39.81071705534969 ...


[Val] Pearson(sample mean/median/std) = 0.9486 / 0.9622 / 0.0415 | R²(sample mean/median/std) = 0.2903 / 0.4062 / 0.5655
[Test] Pearson(sample mean/median/std) = 0.9492 / 0.9623 / 0.0401 | R²(sample mean/median/std) = 0.2563 / 0.3752 / 0.5728
